# Visualize Alignment Optimization Results

This notebook demonstrates how to load and visualize the optimization data captured during tilt-series alignment.

**Use this for:**
- Creating presentation-quality figures
- Exploring optimization behavior
- Analyzing precision evolution
- Comparing subvolumes across optimization steps

In [4]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

from miss_alignment.alignment.visualize_alignment import load_optimization_data

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Optimization Data

In [6]:
# Set path to your optimization tracking directory
tracking_dir = Path("optimization_steps")

# Load all step data
step_data = load_optimization_data(tracking_dir, load_subvolumes=True)

print(f"Loaded {len(step_data)} optimization steps")
print(f"Initial loss: {step_data[0].loss:.6f}")
print(f"Final loss: {step_data[-1].loss:.6f}")
print(f"Loss reduction: {step_data[0].loss - step_data[-1].loss:.6f}")

Loaded 25 optimization steps
Initial loss: -3.274222
Final loss: -4.860538
Loss reduction: 1.586316


## 5. Shift Evolution Analysis

Track how alignment shifts change during optimization.

In [ ]:
if step_data[0].shifts_x is not None:
    # Extract shifts
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)  # (n_steps, n_tilts)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    
    # Compute shift magnitude
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot mean and individual tilts
    mean_magnitude = shift_magnitude.mean(dim=1)
    ax.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean', zorder=10)
    
    for i in range(shifts_x.shape[1]):
        ax.plot(steps, shift_magnitude[:, i], alpha=0.2, color='C0', linewidth=1)
    
    ax.set_xlabel('Optimization Step', fontsize=14)
    ax.set_ylabel('Shift Magnitude (Angstroms)', fontsize=14)
    ax.set_title('Shift Evolution During Optimization', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No shift data available")

## 9. Precision-Weighted Subvolume Grid Visualization

Visualize all subvolumes in a grid, weighted by their precision values to show which patches the model is confident about.

In [ ]:
def create_unweighted_grid(subvolumes, projection_dim):
    """Create a grid of unweighted subvolume projections (reference).

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)  # +1 because first dim is batch

    # Calculate grid dimensions
    n_patches = projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = projections.shape[1], projections.shape[2]

    # Create grid
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = projections[idx].numpy()
        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch

    return grid


def create_precision_weighted_grid(subvolumes, precisions, projection_dim, normalize_precision=True):
    """Create a grid of precision-weighted subvolume projections.

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    precisions : torch.Tensor
        Precision values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    normalize_precision : bool
        Whether to normalize precisions to [0, 1] range.

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)  # +1 because first dim is batch

    # Normalize precisions to [0, 1]
    if normalize_precision:
        precision_min = precisions.min()
        precision_max = precisions.max()
        if precision_max > precision_min:
            precisions_norm = (precisions - precision_min) / (precision_max - precision_min)
        else:
            precisions_norm = torch.ones_like(precisions)
    else:
        precisions_norm = precisions

    # Apply precision weighting
    weighted_projections = projections * precisions_norm.view(-1, 1, 1)

    # Calculate grid dimensions
    n_patches = weighted_projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = weighted_projections.shape[1], weighted_projections.shape[2]

    # Create grid
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = weighted_projections[idx].numpy()
        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch

    return grid


def create_loss_overlay_grid(subvolumes, scores, projection_dim, score_min=None, score_max=None):
    """Create a grid of subvolume projections colored by loss values.

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    scores : torch.Tensor
        Score/loss values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    score_min : float, optional
        Minimum score for normalization. If None, computed from scores.
    score_max : float, optional
        Maximum score for normalization. If None, computed from scores.

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    scores_grid : np.ndarray
        Grid of score values for color overlay.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)

    # Normalize scores using global or local min/max
    if score_min is None:
        score_min = scores.min()
    if score_max is None:
        score_max = scores.max()

    if score_max > score_min:
        scores_norm = (scores - score_min) / (score_max - score_min)
    else:
        scores_norm = torch.ones_like(scores)

    # Calculate grid dimensions
    n_patches = projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = projections.shape[1], projections.shape[2]

    # Create grids
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))
    scores_grid = np.full((grid_rows * patch_h, grid_cols * patch_w), np.nan)

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = projections[idx].numpy()
        score_val = scores_norm[idx].item()

        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch
        scores_grid[row * patch_h:(row + 1) * patch_h,
                   col * patch_w:(col + 1) * patch_w] = score_val

    return grid, scores_grid

### 9.1 Static Precision-Weighted Grid Visualization

Show precision-weighted projections for the first and last optimization steps.

In [ ]:
# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial = create_precision_weighted_grid(
        initial_step.subvolumes,
        initial_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )

    axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear')
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}, Mean Precision: {initial_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')

    # Final step
    grid_final = create_precision_weighted_grid(
        final_step.subvolumes,
        final_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )

    axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear')
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}, Mean Precision: {final_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')

plt.suptitle('Precision-Weighted Subvolume Grid Visualization', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('precision_weighted_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: precision_weighted_grid_comparison.png")

### 9.2 Loss Overlay Grid Visualization

Show subvolume projections with loss values overlaid as a color map.

In [ ]:
# Compute global min/max scores across all steps for consistent normalization
all_scores = torch.cat([s.scores for s in step_data])
global_score_min = all_scores.min().item()
global_score_max = all_scores.max().item()

print(f"Global score range: [{global_score_min:.4f}, {global_score_max:.4f}]")

# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial, scores_grid_initial = create_loss_overlay_grid(
        initial_step.subvolumes,
        initial_step.scores,
        projection_dim=proj_dim,
        score_min=global_score_min,
        score_max=global_score_max
    )

    # Show grayscale projection with loss overlay
    axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear', alpha=0.6)
    im0 = axes[0, proj_dim].imshow(scores_grid_initial, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')
    cbar0 = plt.colorbar(im0, ax=axes[0, proj_dim], fraction=0.046, pad=0.04)
    cbar0.set_label('Normalized Loss', rotation=270, labelpad=20)

    # Final step
    grid_final, scores_grid_final = create_loss_overlay_grid(
        final_step.subvolumes,
        final_step.scores,
        projection_dim=proj_dim,
        score_min=global_score_min,
        score_max=global_score_max
    )

    axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear', alpha=0.6)
    im1 = axes[1, proj_dim].imshow(scores_grid_final, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')
    cbar1 = plt.colorbar(im1, ax=axes[1, proj_dim], fraction=0.046, pad=0.04)
    cbar1.set_label('Normalized Loss', rotation=270, labelpad=20)

plt.suptitle('Loss Overlay on Subvolume Grid (Globally Normalized)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_overlay_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: loss_overlay_grid_comparison.png")

### 9.3 Animated Precision-Weighted Grid

Create animated GIFs showing how precision-weighted subvolume projections evolve during optimization.

In [ ]:
import io
from PIL import Image

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate two-frame comparison animations for each projection direction
for proj_dim in range(3):
    print(f"Generating precision-weighted comparison animation for {projection_names[proj_dim]} projection...")

    # We'll create two-frame GIFs for initial and final steps
    for step_label, step_data_item in [('initial', step_data[0]), ('final', step_data[-1])]:
        # Create reference (unweighted) grid
        grid_reference = create_unweighted_grid(
            step_data_item.subvolumes,
            projection_dim=proj_dim
        )

        # Create precision-weighted grid
        grid_weighted = create_precision_weighted_grid(
            step_data_item.subvolumes,
            step_data_item.precisions,
            projection_dim=proj_dim,
            normalize_precision=True
        )

        # Create two frames
        frames = []

        # Frame 1: Reference (unweighted)
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(grid_reference, cmap='gray', interpolation='bilinear')
        ax.set_title(
            f'Reference (Unweighted) - Step {step_data_item.step}\n'
            f'{projection_names[proj_dim]} Projection\n'
            f'Loss: {step_data_item.loss:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

        # Frame 2: Precision-weighted
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(grid_weighted, cmap='gray', interpolation='bilinear')
        ax.set_title(
            f'Precision-Weighted - Step {step_data_item.step}\n'
            f'{projection_names[proj_dim]} Projection\n'
            f'Loss: {step_data_item.loss:.4f}, Mean Precision: {step_data_item.mean_precision:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

        # Save as two-frame GIF
        if frames:
            output_path = f'precision_weighted_comparison_{projection_names[proj_dim].split()[0].lower()}_{step_label}.gif'
            frames[0].save(
                output_path,
                save_all=True,
                append_images=frames[1:],
                duration=1000,  # 1000ms per frame (1 second)
                loop=0,
                optimize=False,
            )
            print(f"  ✓ Saved: {output_path} (2 frames)")

print("\n✓ All precision-weighted comparison animations complete!")

### 9.4 Animated Loss Overlay Grid

Create animated GIFs showing how loss values across subvolumes evolve during optimization.

In [ ]:
import io
from PIL import Image

# Compute global min/max scores across all steps for consistent normalization
all_scores = torch.cat([s.scores for s in step_data])
global_score_min = all_scores.min().item()
global_score_max = all_scores.max().item()

print(f"Global score range: [{global_score_min:.4f}, {global_score_max:.4f}]\n")

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate animations for each projection direction
for proj_dim in range(3):
    print(f"Generating loss overlay animation for {projection_names[proj_dim]} projection...")

    frames = []

    for step_data_item in step_data:
        # Create grid for this step with global normalization
        grid, scores_grid = create_loss_overlay_grid(
            step_data_item.subvolumes,
            step_data_item.scores,
            projection_dim=proj_dim,
            score_min=global_score_min,
            score_max=global_score_max
        )

        # Create figure
        fig, ax = plt.subplots(figsize=(10, 10))

        # Show grayscale background with loss overlay
        ax.imshow(grid, cmap='gray', interpolation='bilinear', alpha=0.6)
        im = ax.imshow(scores_grid, cmap='coolwarm', interpolation='bilinear', alpha=0.6, vmin=0, vmax=1)

        ax.set_title(
            f'{projection_names[proj_dim]} Projection - Step {step_data_item.step}\n'
            f'Loss: {step_data_item.loss:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Normalized Loss (Global)', rotation=270, labelpad=20)

        plt.tight_layout()

        # Render to image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

    # Save as GIF
    if frames:
        output_path = f'loss_overlay_grid_{projection_names[proj_dim].split()[0].lower()}_evolution.gif'
        frames[0].save(
            output_path,
            save_all=True,
            append_images=frames[1:],
            duration=500,  # 500ms per frame
            loop=0,
            optimize=False,
        )
        print(f"✓ Saved: {output_path} ({len(frames)} frames)")
    else:
        print(f"  No frames generated for {projection_names[proj_dim]} projection")

print("\n✓ All loss overlay animations complete!")

### 9.5 Display Generated Animations

Display the generated GIF animations in the notebook.

In [ ]:
from IPython.display import Image as IPImage, display, HTML
from pathlib import Path

# List of generated GIF files
projection_names = ['depth', 'height', 'width']
gif_types = ['precision_weighted_grid', 'loss_overlay_grid']

print("Generated Animations:\n")

for gif_type in gif_types:
    print(f"\n{'='*60}")
    print(f"{gif_type.replace('_', ' ').title()}")
    print('='*60)

    for proj_name in projection_names:
        gif_path = f'{gif_type}_{proj_name}_evolution.gif'

        if Path(gif_path).exists():
            print(f"\n{proj_name.capitalize()} Projection:")
            display(IPImage(filename=gif_path))
        else:
            print(f"\n{proj_name.capitalize()} Projection: File not found ({gif_path})")

## 10. 4D Tomogram Reconstruction with Overlay Volumes

Reconstruct the full tomogram in 4D (t, d, h, w) where t represents optimization steps, and create overlay volumes for precision, score, and their product.

In [8]:
def reconstruct_4d_volumes_with_overlays(
    step_data_list,
    pixel_size,
    volume_shape=None,
    volume_origin=None,
):
    """Reconstruct 4D volume (t, d, h, w) from subvolumes and create overlay volumes.

    Parameters
    ----------
    step_data_list : list[OptimizationStepData]
        List of optimization step data with subvolumes and positions.
    pixel_size : float
        Pixel size in Angstroms (for converting positions to voxel coordinates).
    volume_shape : tuple[int, int, int] | None
        Target volume shape (d, h, w) in voxels. If None, computed from positions.
    volume_origin : tuple[float, float, float] | None
        Volume origin in Angstroms (x, y, z). If None, computed from positions.

    Returns
    -------
    reconstructed_volume : np.ndarray
        4D volume of shape (n_steps, d, h, w) with reconstructed subvolumes.
    precision_overlay : np.ndarray
        4D overlay volume with normalized precision values.
    score_overlay : np.ndarray
        4D overlay volume with normalized score values.
    precision_score_overlay : np.ndarray
        4D overlay volume with normalized (precision × score) values.
    """
    n_steps = len(step_data_list)
    
    # Get patch size from first subvolume
    patch_d, patch_h, patch_w = step_data_list[0].subvolumes.shape[1:]
    
    # Collect all positions to determine volume bounds
    all_positions = torch.cat([s.positions for s in step_data_list], dim=0)
    
    if volume_origin is None:
        # Compute origin from minimum positions
        # Positions are in Angstroms (x, y, z) at subvolume centers
        min_pos = all_positions.min(dim=0)[0].numpy()
        volume_origin = min_pos - np.array([patch_w, patch_h, patch_d]) * pixel_size / 2
    else:
        volume_origin = np.array(volume_origin)
    
    if volume_shape is None:
        # Compute volume shape from maximum positions
        max_pos = all_positions.max(dim=0)[0].numpy()
        volume_extent = max_pos - volume_origin
        volume_shape = tuple((volume_extent / pixel_size + [patch_w, patch_h, patch_d]).astype(int))
        # Reorder to (d, h, w) from (x, y, z)
        volume_shape = (volume_shape[2], volume_shape[1], volume_shape[0])
    
    print(f"Reconstructing 4D volume with shape: ({n_steps}, {volume_shape[0]}, {volume_shape[1]}, {volume_shape[2]})")
    print(f"Volume origin (x, y, z): {volume_origin}")
    print(f"Pixel size: {pixel_size} Angstroms")
    
    # Initialize volumes
    reconstructed_volume = np.zeros((n_steps, *volume_shape), dtype=np.float32)
    precision_overlay = np.zeros((n_steps, *volume_shape), dtype=np.float32)
    score_overlay = np.zeros((n_steps, *volume_shape), dtype=np.float32)
    precision_score_overlay = np.zeros((n_steps, *volume_shape), dtype=np.float32)
    
    # Count how many subvolumes contribute to each voxel (for proper averaging)
    count_volume = np.zeros((n_steps, *volume_shape), dtype=np.int32)
    
    # Collect all precisions and scores for global normalization
    all_precisions = torch.cat([s.precisions for s in step_data_list], dim=0)
    all_scores = torch.cat([s.scores for s in step_data_list], dim=0)
    
    # Normalize globally
    precision_min = all_precisions.min().item()
    precision_max = all_precisions.max().item()
    precision_range = precision_max - precision_min if precision_max > precision_min else 1.0
    
    score_min = all_scores.min().item()
    score_max = all_scores.max().item()
    score_range = score_max - score_min if score_max > score_min else 1.0
    
    print(f"\\nGlobal precision range: [{precision_min:.6f}, {precision_max:.6f}]")
    print(f"Global score range: [{score_min:.6f}, {score_max:.6f}]")
    
    # Process each step
    for step_idx, step_data in enumerate(step_data_list):
        subvolumes = step_data.subvolumes.numpy()
        positions = step_data.positions.numpy()  # Shape: (n, 3) in (x, y, z) order
        precisions = step_data.precisions.numpy()
        scores = step_data.scores.numpy()
        
        # Normalize precisions and scores
        precisions_norm = (precisions - precision_min) / precision_range
        scores_norm = (scores - score_min) / score_range
        precision_score_norm = precisions_norm * scores_norm
        
        # Place each subvolume
        for i in range(subvolumes.shape[0]):
            subvol = subvolumes[i]  # Shape: (d, h, w)
            pos = positions[i]  # (x, y, z) in Angstroms
            
            # Convert position to voxel coordinates (center of subvolume)
            # Position is (x, y, z), voxel coords are (z, y, x) for (d, h, w)
            voxel_center = ((pos - volume_origin) / pixel_size).astype(int)
            voxel_z, voxel_y, voxel_x = voxel_center[2], voxel_center[1], voxel_center[0]
            
            # Calculate subvolume placement bounds
            z_start = voxel_z - patch_d // 2
            z_end = z_start + patch_d
            y_start = voxel_y - patch_h // 2
            y_end = y_start + patch_h
            x_start = voxel_x - patch_w // 2
            x_end = x_start + patch_w
            
            # Clip to volume bounds
            z_start_clip = max(0, z_start)
            z_end_clip = min(volume_shape[0], z_end)
            y_start_clip = max(0, y_start)
            y_end_clip = min(volume_shape[1], y_end)
            x_start_clip = max(0, x_start)
            x_end_clip = min(volume_shape[2], x_end)
            
            # Skip if completely outside volume
            if (z_start_clip >= z_end_clip or 
                y_start_clip >= y_end_clip or 
                x_start_clip >= x_end_clip):
                continue
            
            # Calculate corresponding subvolume slice
            subvol_z_start = z_start_clip - z_start
            subvol_z_end = subvol_z_start + (z_end_clip - z_start_clip)
            subvol_y_start = y_start_clip - y_start
            subvol_y_end = subvol_y_start + (y_end_clip - y_start_clip)
            subvol_x_start = x_start_clip - x_start
            subvol_x_end = subvol_x_start + (x_end_clip - x_start_clip)
            
            # Place subvolume data
            reconstructed_volume[
                step_idx,
                z_start_clip:z_end_clip,
                y_start_clip:y_end_clip,
                x_start_clip:x_end_clip
            ] += subvol[
                subvol_z_start:subvol_z_end,
                subvol_y_start:subvol_y_end,
                subvol_x_start:subvol_x_end
            ]
            
            # Place constant overlay values
            precision_overlay[
                step_idx,
                z_start_clip:z_end_clip,
                y_start_clip:y_end_clip,
                x_start_clip:x_end_clip
            ] += precisions_norm[i]
            
            score_overlay[
                step_idx,
                z_start_clip:z_end_clip,
                y_start_clip:y_end_clip,
                x_start_clip:x_end_clip
            ] += scores_norm[i]
            
            precision_score_overlay[
                step_idx,
                z_start_clip:z_end_clip,
                y_start_clip:y_end_clip,
                x_start_clip:x_end_clip
            ] += precision_score_norm[i]
            
            # Track overlap count
            count_volume[
                step_idx,
                z_start_clip:z_end_clip,
                y_start_clip:y_end_clip,
                x_start_clip:x_end_clip
            ] += 1
    
    # Average overlapping regions
    mask = count_volume > 0
    reconstructed_volume[mask] /= count_volume[mask]
    precision_overlay[mask] /= count_volume[mask]
    score_overlay[mask] /= count_volume[mask]
    precision_score_overlay[mask] /= count_volume[mask]
    
    # Normalize the precision_score_overlay globally after multiplication
    ps_min = precision_score_overlay[mask].min() if mask.any() else 0
    ps_max = precision_score_overlay[mask].max() if mask.any() else 1
    ps_range = ps_max - ps_min if ps_max > ps_min else 1.0
    precision_score_overlay[mask] = (precision_score_overlay[mask] - ps_min) / ps_range
    
    print(f"\\n✓ Reconstruction complete!")
    print(f"  - Coverage: {mask.sum() / mask.size * 100:.2f}% of voxels have data")
    
    return reconstructed_volume, precision_overlay, score_overlay, precision_score_overlay

### 10.1 Reconstruct 4D Volumes

Reconstruct the full 4D volume and overlay maps from optimization data.

In [9]:
# Set pixel size (Angstroms per pixel)
# You should set this based on your tilt-series data
pixel_size = 10.0  # Example: 10 Angstroms per pixel

# Optional: Specify target volume shape and origin
# If None, these will be computed automatically from positions
volume_shape = None  # e.g., (128, 256, 256) for (d, h, w)
volume_origin = None  # e.g., (0.0, 0.0, 0.0) for (x, y, z) in Angstroms

# Reconstruct 4D volumes
reconstructed_volume, precision_overlay, score_overlay, precision_score_overlay = \
    reconstruct_4d_volumes_with_overlays(
        step_data_list=step_data,
        pixel_size=pixel_size,
        volume_shape=volume_shape,
        volume_origin=volume_origin,
    )

print(f"\nOutput shapes:")
print(f"  Reconstructed volume: {reconstructed_volume.shape}")
print(f"  Precision overlay: {precision_overlay.shape}")
print(f"  Score overlay: {score_overlay.shape}")
print(f"  Precision × Score overlay: {precision_score_overlay.shape}")

Reconstructing 4D volume with shape: (25, 320, 728, 728)
Volume origin (x, y, z): [0. 0. 0.]
Pixel size: 10.0 Angstroms
\nGlobal precision range: [0.034917, 0.412694]
Global score range: [-8.686689, 3.945567]
\n✓ Reconstruction complete!
  - Coverage: 37.64% of voxels have data

Output shapes:
  Reconstructed volume: (25, 320, 728, 728)
  Precision overlay: (25, 320, 728, 728)
  Score overlay: (25, 320, 728, 728)
  Precision × Score overlay: (25, 320, 728, 728)


### 10.2 Save 4D Volumes to Disk

Save the reconstructed volumes as numpy arrays for viewing in external tools.

In [10]:
# Define output directory
output_dir = Path("4d_reconstruction_output")
output_dir.mkdir(exist_ok=True)

# Save metadata
metadata = {
    "n_steps": reconstructed_volume.shape[0],
    "volume_shape": reconstructed_volume.shape[1:],
    "pixel_size": pixel_size,
    "description": {
        "reconstructed_volume": "4D volume (t, d, h, w) with reconstructed subvolumes",
        "precision_overlay": "4D overlay with normalized precision values (0-1)",
        "score_overlay": "4D overlay with normalized score values (0-1)",
        "precision_score_overlay": "4D overlay with normalized (precision × score) values (0-1)",
    }
}

import json
with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved 4D volumes to: {output_dir}/")
print(f"  - reconstructed_volume.npy: {reconstructed_volume.shape}")
print(f"  - precision_overlay.npy: {precision_overlay.shape}")
print(f"  - score_overlay.npy: {score_overlay.shape}")
print(f"  - precision_score_overlay.npy: {precision_score_overlay.shape}")
print(f"  - metadata.json")

✓ Saved 4D volumes to: 4d_reconstruction_output/
  - reconstructed_volume.npy: (25, 320, 728, 728)
  - precision_overlay.npy: (25, 320, 728, 728)
  - score_overlay.npy: (25, 320, 728, 728)
  - precision_score_overlay.npy: (25, 320, 728, 728)
  - metadata.json


In [17]:
def f(projection, weight):
    projection[weight > 0] /= weight[weight > 0]
    return projection

weights = (precision_overlay != 0) * 1
weights = weights.sum(axis=(-3))
projection = f(reconstructed_volume.sum(axis=(-3)), weights)
projection_precision = f(precision_overlay.sum(axis=(-3)), weights)
projection_score = f(score_overlay.sum(axis=(-3)), weights)
projection_precision_score = f(precision_score_overlay.sum(axis=(-3)), weights)

np.save(output_dir / "projected_volume.npy", projection)
np.save(output_dir / "projected_precision_overlay.npy", projection_precision)
np.save(output_dir / "projected_score_overlay.npy", projection_score)
np.save(output_dir / "projected_precision_score_overlay.npy", projection_precision_score)